# Room Temperature Model
### Thermostat control and climate interaction

This model simulates the indoor temperature of a room controlled by a thermostat while interacting with a time-varying outdoor climate. It demonstrates the balance between active heating (to reach a setpoint) and passive heat loss (to the environment).

## How it works

At each time step, the indoor temperature is updated based on two factors:

| Component | Formula | Description |
|:---|:---|:---|
| Heating | $\theta \cdot (T_{set} - T_{in})$ | Active heating towards the setpoint |
| Loss | $\lambda \cdot (T_{in} - T_{out})$ | Passive heat exchange with the outside |

The outdoor temperature $T_{out}$ follows a midday-peak curve by default.

## Imports

In [ ]:
from dissmodel.core import Environment
from dissmodel_sysdyn.models import RoomTemperature
from dissmodel.visualization import Chart

## ⚠️ Important: instantiation order

The `Environment` must always be created **before** any model.

```
Environment  →  RoomTemperature  →  Chart
```

In [ ]:
env = Environment()

## Setting up the model

The Room Temperature model accepts the following parameters:

| Parameter | Description | Default |
|:---|:---|:---|
| `temp_set` | Thermostat setpoint (°C) | 20.0 |
| `inside` | Initial indoor temperature (°C) | 15.0 |
| `outside` | Initial outdoor temperature (°C) | 1.0 |
| `thermal_inertia` | Heating speed factor $\theta$ | 0.33 |
| `loss_to_outside` | Loss rate factor $\lambda$ | 0.30 |

In [ ]:
from dissmodel.core import Model
from dissmodel.visualization import track_plot

@track_plot("inside", "orange")
@track_plot("outside", "blue")
@track_plot("temp_set", "red")
class RoomTemperature(Model):
    def setup(self, temp_set=20.0, inside=15.0, outside=1.0, thermal_inertia=0.33, loss_to_outside=0.30, climate=None) -> None:
        self.temp_set, self.inside, self.outside = temp_set, inside, outside
        self.thermal_inertia, self.loss_to_outside = thermal_inertia, loss_to_outside
        self._climate = climate if climate else (lambda t: 15.0 - 0.1 * (t - 12.0) ** 2)

    def execute(self) -> None:
        t = self.env.now()
        self.outside = self._climate(t)
        inflow  = self.thermal_inertia * (self.temp_set - self.inside)
        outflow = self.loss_to_outside  * (self.inside - self.outside)
        self.inside += inflow - outflow

In [ ]:
RoomTemperature(temp_set=22.0, inside=15.0)

## Visualization

The chart will show the interaction between the fixed setpoint, the fluctuating outdoor temperature, and the responding indoor temperature.

In [ ]:
Chart(show_legend=True, show_grid=True, title="Thermostat Control")

## Running the simulation

In [ ]:
env.run(24)

## Guided Experiments

- **Better Insulation**: Decrease `loss_to_outside` to 0.05 and see how much more stable the indoor temperature becomes.
- **Powerful Heater**: Increase `thermal_inertia` to 0.8. How quickly does the room heat up?
- **Extreme Climate**: Define a custom climate function, e.g., `lambda t: -10 if t < 12 else 0`.
- **Lower Setpoint**: Change `temp_set` to 18.0.

## Conclusion

The Room Temperature model illustrates a classic control problem. By adjusting the parameters of the negative feedback loop (the thermostat), we can maintain a stable internal state even when faced with external disturbances.